# Existing Log Evaluation and Loss Visualization

`/home/carol/chaeyeon-kim/alpha26/logs` 아래에 이미 저장된 Lightning/TensorBoard 로그를 읽어서 학습 loss, validation loss, sensor별 reconstruction loss, 그리고 새로 추가된 PSNR/Chamfer metric 태그가 있을 경우 평가 그래프까지 자동으로 시각화합니다.

과거 로그에는 `PSNR`/`Chamfer` 태그가 없을 수 있으므로, 이 노트북은 기존 `val_rgb_*`, `val_lidar_*` loss를 fallback 평가 지표로 함께 보여줍니다.


In [ ]:
from pathlib import Path
import re
import math

import pandas as pd
import matplotlib.pyplot as plt

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "tensorboard 패키지가 필요합니다. 현재 환경에서 `pip install tensorboard` 후 다시 실행하세요."
    ) from exc

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

ROOT = Path("/home/carol/chaeyeon-kim/alpha26")
LOG_ROOT = ROOT / "logs"
LIGHTNING_LOG_DIR = LOG_ROOT / "lightning_logs"
FIG_DIR = ROOT / "log_figures"
FIG_DIR.mkdir(exist_ok=True)

if not LIGHTNING_LOG_DIR.exists():
    raise FileNotFoundError(f"Lightning log directory not found: {LIGHTNING_LOG_DIR}")


def version_number(path_or_name):
    name = Path(path_or_name).name
    match = re.search(r"version_(\d+)", name)
    return int(match.group(1)) if match else -1


def smooth_series(values, window=25):
    return values.rolling(window, min_periods=1).mean()


def select_tags(df, include=(), exclude=()):
    tags = sorted(df["tag"].unique())
    selected = []
    for tag in tags:
        if include and not any(token in tag for token in include):
            continue
        if exclude and any(token in tag for token in exclude):
            continue
        selected.append(tag)
    return selected


In [ ]:
rows = []
tag_summary = []
versions = sorted(LIGHTNING_LOG_DIR.glob("version_*"), key=version_number)

for version_dir in versions:
    event_files = sorted(version_dir.glob("events.out.tfevents.*"))
    if not event_files:
        continue

    accumulator = EventAccumulator(str(version_dir), size_guidance={"scalars": 0})
    accumulator.Reload()
    scalar_tags = sorted(accumulator.Tags().get("scalars", []))

    tag_summary.append({
        "version": version_dir.name,
        "version_num": version_number(version_dir),
        "event_files": len(event_files),
        "scalar_tags": len(scalar_tags),
        "first_step": None,
        "last_step": None,
        "tags": ", ".join(scalar_tags),
    })

    for tag in scalar_tags:
        events = accumulator.Scalars(tag)
        for event in events:
            rows.append({
                "version": version_dir.name,
                "version_num": version_number(version_dir),
                "tag": tag,
                "step": event.step,
                "wall_time": event.wall_time,
                "value": event.value,
            })

df = pd.DataFrame(rows)
tag_df = pd.DataFrame(tag_summary)

if df.empty:
    raise ValueError(f"No TensorBoard scalar data found in {LIGHTNING_LOG_DIR}")

step_summary = df.groupby("version")["step"].agg(["min", "max"]).reset_index()
tag_df = tag_df.merge(step_summary, on="version", how="left").rename(columns={"min": "first_step", "max": "last_step"})

latest_version = f"version_{df['version_num'].max()}"
print(f"Loaded {len(df):,} scalar rows from {len(tag_df)} versions. Latest: {latest_version}")
display(tag_df[["version", "scalar_tags", "first_step", "last_step"]])


In [ ]:
def plot_tags(tags, version=None, title=None, ylabel="value", smooth_window=25, figsize=(14, 7), save_name=None):
    tags = [tag for tag in tags if tag in set(df["tag"])]
    if not tags:
        print("No matching tags found.")
        return None

    plot_df = df[df["tag"].isin(tags)].copy()
    if version is not None:
        plot_df = plot_df[plot_df["version"].eq(version)].copy()
    if plot_df.empty:
        print("No rows to plot after filtering.")
        return None

    plot_df = plot_df.sort_values(["version_num", "tag", "step"])
    plot_df["smoothed"] = plot_df.groupby(["version", "tag"])["value"].transform(
        lambda s: smooth_series(s, smooth_window)
    )

    fig, ax = plt.subplots(figsize=figsize)
    for (version_name, tag), group in plot_df.groupby(["version", "tag"], sort=True):
        label = tag if version is not None else f"{version_name}:{tag}"
        ax.plot(group["step"], group["smoothed"], label=label, linewidth=1.8)

    ax.set_title(title or ", ".join(tags))
    ax.set_xlabel("step")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_name:
        out = FIG_DIR / save_name
        fig.savefig(out, dpi=160, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    return fig


def plot_tag_groups(tag_groups, version=None, title="", smooth_window=25, figsize=(16, 10), save_name=None):
    n = len(tag_groups)
    cols = 2
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = list(pd.Series(axes.ravel()))

    for ax, (panel_title, tags, ylabel) in zip(axes, tag_groups):
        tags = [tag for tag in tags if tag in set(df["tag"])]
        if not tags:
            ax.text(0.5, 0.5, "No matching scalar", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(panel_title)
            ax.axis("off")
            continue

        plot_df = df[df["tag"].isin(tags)].copy()
        if version is not None:
            plot_df = plot_df[plot_df["version"].eq(version)].copy()
        plot_df = plot_df.sort_values(["version_num", "tag", "step"])
        plot_df["smoothed"] = plot_df.groupby(["version", "tag"])["value"].transform(
            lambda s: smooth_series(s, smooth_window)
        )
        for (version_name, tag), group in plot_df.groupby(["version", "tag"], sort=True):
            label = tag if version is not None else f"{version_name}:{tag}"
            ax.plot(group["step"], group["smoothed"], label=label, linewidth=1.7)
        ax.set_title(panel_title)
        ax.set_xlabel("step")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title, y=1.02, fontsize=15)
    plt.tight_layout()
    if save_name:
        out = FIG_DIR / save_name
        fig.savefig(out, dpi=160, bbox_inches="tight")
        print(f"Saved: {out}")
    plt.show()
    return fig


## 1. 전체 Run의 Main Loss 비교

In [ ]:
main_loss_tags = [tag for tag in ["train_loss_step", "train_loss_epoch", "val_loss", "val/RL_loss", "val/DS_loss"] if tag in set(df["tag"])]
plot_tags(
    main_loss_tags,
    version=None,
    title="Main train/validation loss across all logged versions",
    ylabel="loss",
    smooth_window=25,
    figsize=(16, 8),
    save_name="all_versions_main_loss.png",
)


## 2. 최신 Run의 Loss Component

In [ ]:
latest_loss_tags = [
    tag for tag in sorted(df["tag"].unique())
    if df[df["version"].eq(latest_version)]["tag"].eq(tag).any()
    and (tag.startswith("train_") or tag.startswith("val_") or tag.startswith("val/"))
    and ("loss" in tag or "rgb" in tag or "lidar" in tag or "probabilistic" in tag)
    and not any(metric in tag for metric in ["psnr", "chamfer", "range_mae", "xyz_euclidean"])
]

plot_tags(
    latest_loss_tags,
    version=latest_version,
    title=f"Loss components in {latest_version}",
    ylabel="loss",
    smooth_window=25,
    figsize=(16, 9),
    save_name=f"{latest_version}_loss_components.png",
)


## 3. 기존 로그 기반 평가 그래프

과거 로그에는 PSNR/Chamfer가 직접 저장되어 있지 않으므로, 기존 validation reconstruction loss를 sensor별 평가 proxy로 시각화합니다. 값은 낮을수록 좋습니다.

In [ ]:
legacy_eval_groups = [
    ("RGB reconstruction loss", ["val_rgb_2", "val_rgb_4", "val_future_rgb_2", "val_future_rgb_4"], "L1 loss"),
    ("LiDAR XYZ reconstruction loss", ["val_lidar_xyz_2", "val_lidar_xyz_4", "val_future_lidar_xyz_2", "val_future_lidar_xyz_4"], "MSE loss"),
    ("LiDAR range/depth reconstruction loss", ["val_lidar_depth_2", "val_lidar_depth_4", "val_future_lidar_depth_2", "val_future_lidar_depth_4"], "L1 loss"),
    ("Probabilistic KL", ["val_probabilistic", "train_probabilistic_epoch", "train_probabilistic_step"], "KL loss"),
]

plot_tag_groups(
    legacy_eval_groups,
    version=latest_version,
    title=f"Evaluation/loss proxy metrics from existing logs: {latest_version}",
    smooth_window=5,
    figsize=(16, 10),
    save_name=f"{latest_version}_legacy_eval_proxy.png",
)


## 4. 새 Metric 태그 자동 시각화

`trainer11.ipynb`에 새로 추가한 평가 코드로 이후 학습/검증을 실행하면 `rgb_psnr`, `lidar_chamfer_xyz`, `lidar_xyz_euclidean`, `lidar_range_mae` 태그가 생깁니다. 현재 로그에 해당 태그가 있으면 자동으로 표시합니다.

In [ ]:
metric_groups = [
    ("Camera PSNR: higher is better", select_tags(df, include=("rgb_psnr",)), "PSNR (dB)"),
    ("LiDAR Chamfer Distance: lower is better", select_tags(df, include=("chamfer",)), "Chamfer distance"),
    ("LiDAR XYZ Euclidean Error: lower is better", select_tags(df, include=("xyz_euclidean",)), "meters"),
    ("LiDAR Range MAE: lower is better", select_tags(df, include=("range_mae",)), "meters"),
]

plot_tag_groups(
    metric_groups,
    version=latest_version,
    title=f"New evaluation metrics if present: {latest_version}",
    smooth_window=5,
    figsize=(16, 10),
    save_name=f"{latest_version}_new_eval_metrics.png",
)


## 5. MUVO 논문 Figure 스타일 4-panel

새 로그에 `val/RL_*`, `val/DS_*` 태그가 있으면 논문 정리처럼 RL/DS x RGB/LiDAR 4개 패널로 표시합니다. 현재 과거 로그처럼 RL/DS 분리 태그가 없으면 자동으로 안내 문구가 나옵니다.

In [ ]:
muvo_style_groups = [
    ("D_val^RL RGB prediction", ["val/RL_rgb_psnr", "val/RL_future_rgb_psnr"], "PSNR (dB)"),
    ("D_val^DS RGB prediction", ["val/DS_rgb_psnr", "val/DS_future_rgb_psnr"], "PSNR (dB)"),
    ("D_val^RL LiDAR prediction", ["val/RL_lidar_chamfer_xyz", "val/RL_future_lidar_chamfer_xyz"], "Chamfer distance"),
    ("D_val^DS LiDAR prediction", ["val/DS_lidar_chamfer_xyz", "val/DS_future_lidar_chamfer_xyz"], "Chamfer distance"),
]

plot_tag_groups(
    muvo_style_groups,
    version=latest_version,
    title=f"MUVO-style RL/DS evaluation panels: {latest_version}",
    smooth_window=5,
    figsize=(16, 10),
    save_name=f"{latest_version}_muvo_style_eval.png",
)


## 6. 원하는 태그 직접 선택

In [ ]:
available_tags = pd.DataFrame({"tag": sorted(df["tag"].unique())})
display(available_tags)

CUSTOM_TAGS = [
    "train_loss_step",
    "val_loss",
]

plot_tags(
    CUSTOM_TAGS,
    version=latest_version,
    title=f"Custom tags: {latest_version}",
    ylabel="value",
    smooth_window=25,
    figsize=(14, 7),
)
